# Cellpose-SAM 細胞ROI抽出 (Google Colab版)

研究室のGPU付きPCが故障したため、[Cellpose公式のColabノートブック](https://github.com/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb)（Cellpose-SAM）をベースに、この研究室の環境に合わせて構成したノートブックです。

- Cellpose本体は最新の **Cellpose-SAM**（`models.CellposeModel`）を使用（細胞の直径指定は不要）
- 撮影データはZスタック（`_C001Z001` 〜 `_C001Z007` のようなZスライスtifが複数枚、撮影ごとにフォルダが分かれている）想定 → **Zスライスごとに個別にセグメンテーション・ROI作成**（Z投影はしない。ピントの良いZは後でFiji上で選ぶ）
- セグメンテーション結果は、元のtifと**同じフォルダ・同じファイル名の `.zip`**（ImageJ RoiSet形式）として保存 → 従来の `intensity_analysis.ijm` マクロがそのまま読み込める

## Cellposeの設定値（研究室PCでの設定に合わせています）

- `flow_threshold` = 0.1
- `cellprob_threshold` = 2.0
- 最小面積 100 µm² / 最大面積 500 µm² → 元の設定はµm²単位とみられるため、各tifのピクセルサイズ(µm/px)を使って画像ごとにpx換算してから、Cellposeの `min_size` パラメータ・このノートブック側の後処理フィルタにそれぞれ適用
- Max Brightness Ratio 2.5 → **Cellpose本体には該当する設定がなく、正確な計算式が未確認のため、デフォルトでは無効化したプレースホルダーとして実装しています**（下記パラメータ設定セル参照）。有効化する前に、元のツールでの正確な定義をご確認ください。

## 全体の流れ

1. (このノートブック) Google Drive上の親フォルダ以下を再帰的に探索し、すべての `.tif` / `.tiff`（＝各撮影フォルダの各Zスライス）を読み込む
2. (このノートブック) Zスライス1枚ずつをCellpose-SAMで自動セグメンテーションし、面積フィルタ等を適用する
3. (このノートブック) セグメンテーション結果を、元のtifと同じフォルダ・同じファイル名の `.zip`（ImageJ RoiSet形式）として保存する
4. (Fiji/ImageJ) 使うZを選び、その `画像名.zip` を対応する画像と一緒に開き、ROI Managerでおかしい細胞のROIを削除し、背景ROIを最後に追加してzipを保存し直す
5. (Fiji/ImageJ) 従来の `intensity_analysis.ijm` マクロを実行して膜/細胞質の輝度解析を行う

## 事前準備

- 上部メニューの **「ランタイム」→「ランタイムのタイプを変更」→ハードウェアアクセラレータで GPU (T4など) を選択**してください。
- 撮影ごとのフォルダ（各フォルダの中にZスライスのtifが入っている）を、Google Drive上の1つの親フォルダにまとめておいてください。サブフォルダ構成のままで構いません（このノートブックが再帰的に探索します）。


In [ ]:
# GPUが割り当てられているか確認
!nvidia-smi


In [ ]:
# Cellposeのインストール（Cellpose-SAMを含む最新版。natsort/tifffile/roifile等も依存関係として入る）
!pip install -q cellpose


**注意**: インストールでColabに元から入っているnumpyが更新され、そのままだと
`ImportError: cannot import name '_center' from 'numpy._core.umath'` のような
numpyのバージョン不整合エラーが出ることがあります。

これを避けるため、次のセルでランタイムを一度再起動します。実行すると
「セッションがクラッシュしました」のような表示が出ますが正常な動作です。
再起動後、**上のインストールセルは再実行せず**、この次の「Google Driveをマウント」の
セルから続けて実行してください。


In [ ]:
# ランタイムを再起動して、更新されたnumpyを正しく読み込み直す
import os
os.kill(os.getpid(), 9)


In [ ]:
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')


### モデルの重みをGoogle Driveにキャッシュする（次回以降を高速化）

Cellpose-SAMのモデルの重み（数百MB程度）は、初回は必ずダウンロードに時間がかかります。
Colabのランタイムは毎回まっさらな状態にリセットされるため、何もしないと**セッションを
開き直すたびに毎回ダウンロードし直す**ことになります。

そこで、保存先をGoogle Drive上のフォルダに固定します。これで一度ダウンロードすれば、
次回以降のセッションではDriveから読み込むだけで済み、モデル準備がすぐに終わります。
**このセルは必ず、下で `from cellpose import models` を実行するより前に実行してください**
（`cellpose`を一度importすると、保存先の設定が固定されてしまうため）。


In [ ]:
import os

# モデルの重みをここに保存する（Google Drive上なのでセッションをまたいで保持される）
CELLPOSE_MODEL_CACHE_DIR = "/content/drive/MyDrive/cellpose_models_cache"
os.makedirs(CELLPOSE_MODEL_CACHE_DIR, exist_ok=True)

# cellposeをimportする前にこの環境変数を設定しておくことで、
# cellposeがモデルの重みを探す/保存する場所がこのフォルダになる
os.environ["CELLPOSE_LOCAL_MODELS_PATH"] = CELLPOSE_MODEL_CACHE_DIR


## パス設定・画像の探索

`INPUT_DIR` を、撮影フォルダ（Zスライスtifが入ったフォルダ）をまとめてある親フォルダに変更してください。例えば

```
INPUT_DIR/
  260703-WT-PMA-60min-1408-488/
    260703-WT-PMA-60min-1408-488_C001Z001.tif
    ...
    260703-WT-PMA-60min-1408-488_C001Z007.tif
  260703-他の撮影/
    ...
```

のような構成を想定し、`INPUT_DIR` 以下を再帰的に探索してすべての `.tif`/`.tiff` を1枚ずつ処理します（Z投影はせず、Zスライスごとに個別にROIを作ります）。

`OUTPUT_DIR` は ROI (`.zip`) の保存先です。Fijiのマクロは「画像と同じフォルダにある同名の `.zip`」を探すので、迷ったら `INPUT_DIR` と同じにしておくのが安全です（デフォルトでそうなっています。各tifと同じサブフォルダに保存されます）。

もし1つの撮影フォルダに複数チャンネル（例: `C001`, `C002`）のtifが混在していて、セグメンテーションに使いたいチャンネルが1つだけの場合は、`FILENAME_FILTER` にそのチャンネルを表す文字列（例: `"C001"`）を指定すると、そのチャンネルのファイルだけを処理できます。


In [ ]:
from pathlib import Path
from natsort import natsorted

# 撮影ごとのフォルダ（Zスライスtifが入っている）をまとめた親フォルダ
INPUT_DIR = Path("/content/drive/MyDrive/cellpose_input")
if not INPUT_DIR.exists():
    raise FileNotFoundError("INPUT_DIRが存在しません。パスを確認してください。")

# ROI (.zip) の保存先。Fijiのマクロが「各tifと同じフォルダの同名zip」を探すため、
# 基本はINPUT_DIRと同じにしておき、各tifと同じサブフォルダに保存する
OUTPUT_DIR = INPUT_DIR

# セグメンテーション結果を目視確認するための重ね合わせ画像 (QC画像) の保存先
# (INPUT_DIR以下のフォルダ構成をそのままミラーして保存する)
QC_DIR = OUTPUT_DIR / "qc_overlays"

# 複数チャンネルがあり、特定のチャンネルだけをセグメントしたい場合はファイル名に
# 含まれる文字列を指定する（例: "C001"）。すべて処理する場合は None のままでよい。
FILENAME_FILTER = None

QC_DIR.mkdir(parents=True, exist_ok=True)

# INPUT_DIR以下を再帰的に探索し、INPUT_DIRからの相対パスを集める
image_files = [
    p.relative_to(INPUT_DIR)
    for p in INPUT_DIR.rglob("*")
    if p.suffix.lower() in (".tif", ".tiff")
    and QC_DIR not in p.parents
    and (FILENAME_FILTER is None or FILENAME_FILTER in p.name)
]
image_files = natsorted(image_files, key=lambda p: str(p))

if len(image_files) == 0:
    raise FileNotFoundError("画像が見つかりませんでした。INPUT_DIRやFILENAME_FILTERを確認してください。")

print(f"{len(image_files)} 枚のZスライス画像が見つかりました（1枚ずつ個別にROIを作成します）")
for f in image_files:
    print(" -", f)


## モデルの準備

Cellpose-SAM（`models.CellposeModel`）を読み込みます。旧来のcyto2/cyto3のようなモデル選択や細胞直径の指定は不要です（初回実行時に重みがダウンロードされます）。

**注意**: Cellpose内蔵のダウンローダーは、Hugging Face側のbot対策により
`HTTPError: HTTP Error 403: Forbidden` で失敗することがあります。そのため、
次のセルで先に重みをダウンロードしておき、Cellpose自身のダウンロード処理をスキップさせます。
匿名アクセスは速度制限や途中停止が起きやすいため、**途中から再開できるリトライ処理付き**の
ダウンロード関数にしています。途中で止まったように見えても、そのまま待つか、
セルを再実行すれば続きからダウンロードされます。


In [ ]:
# Cellpose内蔵のダウンローダーが403 Forbiddenで失敗することがあるのに加えて、
# HuggingFaceの匿名アクセスは速度制限で途中停止しやすいため、
# requestsで直接、レジューム(途中再開)・リトライ付きでダウンロードする
# (保存先は上で設定した CELLPOSE_LOCAL_MODELS_PATH = Google Drive上のフォルダ)
import time
from pathlib import Path
import requests
from tqdm.auto import tqdm

CELLPOSE_MODEL_DIR = Path(os.environ["CELLPOSE_LOCAL_MODELS_PATH"])
CELLPOSE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
cached_weight_path = CELLPOSE_MODEL_DIR / "cpsam_v2"
MODEL_URL = "https://huggingface.co/mouseland/cellpose-sam/resolve/main/cpsam_v2"


def download_with_resume(url, dst, max_retries=8):
    headers = {"User-Agent": "Mozilla/5.0 (Colab; cellpose-roi-colab notebook)"}
    tmp_path = dst.with_name(dst.name + ".part")

    for attempt in range(1, max_retries + 1):
        resume_pos = tmp_path.stat().st_size if tmp_path.exists() else 0
        req_headers = dict(headers)
        if resume_pos:
            req_headers["Range"] = f"bytes={resume_pos}-"
        try:
            with requests.get(url, headers=req_headers, stream=True, timeout=60) as r:
                if r.status_code not in (200, 206):
                    r.raise_for_status()
                total = int(r.headers.get("Content-Length", 0)) + resume_pos
                mode = "ab" if resume_pos else "wb"
                with open(tmp_path, mode) as f, tqdm(
                    total=total, initial=resume_pos, unit="B", unit_scale=True,
                    unit_divisor=1024, desc="cpsam_v2"
                ) as pbar:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))
            tmp_path.rename(dst)
            return
        except Exception as e:
            print(f"ダウンロードが途中で止まりました（試行 {attempt}/{max_retries}）: {e}")
            print("少し待ってから続きからダウンロードを再開します...")
            time.sleep(10)

    raise RuntimeError(
        "モデルの重みのダウンロードに繰り返し失敗しました。時間をおいてこのセルを再実行してください。"
    )


if not cached_weight_path.exists():
    print("モデルの重みをダウンロードします（初回のみ、数百MB〜1GB程度）...")
    download_with_resume(MODEL_URL, cached_weight_path)
    print(f"モデルの重みをダウンロードしました: {cached_weight_path}")
else:
    print(f"モデルの重みは既にDriveにキャッシュされています（再ダウンロード不要）: {cached_weight_path}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cellpose import models, core, io, utils, plot

io.logger_setup()  # 進捗ログを表示する

if core.use_gpu() == False:
    raise ImportError("GPUにアクセスできません。「ランタイム」→「ランタイムのタイプを変更」でGPUを選択してください。")

model = models.CellposeModel(gpu=True)


## パラメータ設定

研究室のPCで使っていた設定値です。

- `FLOW_THRESHOLD` (0.1) / `CELLPROB_THRESHOLD` (2.0): Cellposeの `model.eval()` にそのまま渡します
- `MIN_AREA_UM2` (最小面積 100 µm²) / `MAX_AREA_UM2` (最大面積 500 µm²): 研究室PCでの元の設定は**µm²単位**だったとみられるため、µm²で指定します。実際にCellposeへ渡すpx単位の値は、各tifのメタデータから読み取ったピクセルサイズ(µm/px)を使って画像ごとに自動換算します（下のセル参照）
- `TILE_NORM_BLOCKSIZE`: 画像内で明るさが大きく不均一な場合は100〜200程度に変更（デフォルト0は画像全体を一括で正規化）

**面積の単位について**: 当初 `MIN_SIZE=100px` / `MAX_AREA=500px` として実行したところ、Cellpose検出79個 → MAX_AREAフィルタ後8個まで激減する事例がありました。実測したところ画像は「212.13×212.13 µm (512×512px)」、つまり約0.4143 µm/pxで、500pxはこの解像度では明らかに小さすぎる値でした。おそらく元の「最小面積100・最大面積500」はµm²指定だったため、ここではµm²をピクセルサイズで換算してからCellposeに渡す方式にしています。

**Max Brightness Ratio (2.5) について**: Cellpose本体には該当する設定がなく、元のツールでの正確な計算式が未確認です。誤ったROIの採否につながらないよう、デフォルトでは `APPLY_BRIGHTNESS_RATIO_FILTER = False` にして無効化しています。下のセルの `filter_by_brightness_ratio()` は「ROI内の最大輝度／平均輝度の比」という仮実装のプレースホルダーです。元のツールでの正確な定義を確認できたら、この関数を修正した上で `True` にしてください。

**検出される細胞数が明らかに少ないと感じる場合**: `segment_and_filter()` はフィルタ前後の内訳（Cellpose検出直後の個数 → 最大面積フィルタ後の個数）と、使われたピクセルサイズ・px換算後の面積を表示するようにしているので、まずそれで確認してください。`CELLPROB_THRESHOLD = 2.0` もCellposeのデフォルト値 `0.0` よりかなり厳しい設定なので、少ないと感じる場合はこちらも下げて試してみてください。


In [ ]:
FLOW_THRESHOLD = 0.1
CELLPROB_THRESHOLD = 2.0
MIN_AREA_UM2 = 100            # 最小面積(um^2)。研究室PCでの元の設定値
MAX_AREA_UM2 = 500            # 最大面積(um^2)。研究室PCでの元の設定値
TILE_NORM_BLOCKSIZE = 0       # 明るさが不均一な画像なら100-200程度に変更

# tifのメタデータからピクセルサイズ(um/px)を自動取得できなかった場合に使うフォールバック値。
# 実測値の例: 212.13um / 512px = 0.4143 um/px
FALLBACK_PIXEL_SIZE_UM = 212.13 / 512

# Trueにすると、tifのメタデータを無視して常にFALLBACK_PIXEL_SIZE_UMを使う
FORCE_FALLBACK_PIXEL_SIZE = False

# Max Brightness Ratio: 正確な定義が未確認のため、デフォルトでは無効
APPLY_BRIGHTNESS_RATIO_FILTER = False
MAX_BRIGHTNESS_RATIO = 2.5


## 面積フィルタ・輝度比フィルタ・ROI保存の関数


In [ ]:
import roifile
import tifffile


def get_pixel_size_um(tif_path):
    """tifのメタデータからピクセルサイズ(um/px)を取得する。取得できない場合はNoneを返す"""
    try:
        with tifffile.TiffFile(str(tif_path)) as tf:
            page = tf.pages[0]
            xres_tag = page.tags.get("XResolution")
            if xres_tag is None:
                return None
            num, den = xres_tag.value
            if num == 0:
                return None
            px_per_unit = num / den  # 1単位あたりのピクセル数

            # ImageJ形式のメタデータ(unit)があれば優先的に使う
            ij_meta = tf.imagej_metadata or {}
            unit = str(ij_meta.get("unit", "")).lower()
            if unit in ("um", "micron", "microns", "µm", "micrometer"):
                return 1.0 / px_per_unit

            unit_tag = page.tags.get("ResolutionUnit")
            unit_value = unit_tag.value if unit_tag is not None else 2
            if unit_value == 3:  # センチメートル
                return 1.0e4 / px_per_unit
            elif unit_value == 2:  # インチ
                return 25400.0 / px_per_unit
    except Exception:
        return None
    return None


def remove_large_masks(masks, max_area_px):
    """max_area_px(px)を超えるROIを除去し、ラベルを振り直す"""
    labels, counts = np.unique(masks, return_counts=True)
    keep_labels = [l for l, c in zip(labels, counts) if l != 0 and c <= max_area_px]

    new_masks = np.zeros_like(masks)
    for new_id, old_id in enumerate(keep_labels, start=1):
        new_masks[masks == old_id] = new_id
    return new_masks


def filter_by_brightness_ratio(masks, img, max_ratio):
    """TODO: 元ツールでの正確な定義が未確認のプレースホルダー実装。
    現状の仮実装: ROI内の (最大輝度 / 平均輝度) が max_ratio を超えるROIを除外する。
    """
    labels = np.unique(masks)
    labels = labels[labels != 0]
    keep_labels = []
    for l in labels:
        pixels = img[masks == l]
        mean_val = pixels.mean()
        if mean_val <= 0:
            continue
        ratio = pixels.max() / mean_val
        if ratio <= max_ratio:
            keep_labels.append(l)

    new_masks = np.zeros_like(masks)
    for new_id, old_id in enumerate(keep_labels, start=1):
        new_masks[masks == old_id] = new_id
    return new_masks


def segment_and_filter(img, tif_path, verbose=False):
    """Cellpose-SAMでセグメンテーションし、面積・輝度比フィルタを適用する"""
    pixel_size_um = None if FORCE_FALLBACK_PIXEL_SIZE else get_pixel_size_um(tif_path)
    used_fallback = pixel_size_um is None
    if used_fallback:
        pixel_size_um = FALLBACK_PIXEL_SIZE_UM

    min_size_px = max(1, int(round(MIN_AREA_UM2 / (pixel_size_um ** 2))))
    max_area_px = MAX_AREA_UM2 / (pixel_size_um ** 2)

    if verbose:
        source = "フォールバック値" if used_fallback else "tifのメタデータから自動取得"
        print(
            f"  ピクセルサイズ: {pixel_size_um:.4f} um/px ({source}) → "
            f"最小面積{min_size_px}px・最大面積{max_area_px:.0f}px として適用"
        )

    masks, flows, styles = model.eval(
        img,
        batch_size=32,
        flow_threshold=FLOW_THRESHOLD,
        cellprob_threshold=CELLPROB_THRESHOLD,
        min_size=min_size_px,
        normalize={"tile_norm_blocksize": TILE_NORM_BLOCKSIZE},
    )
    n_raw = int(masks.max())

    if verbose and n_raw > 0:
        # Cellpose検出直後（面積フィルタをかける前）のROI面積(px)の分布。
        # MAX_AREA_UM2が実際の細胞サイズに対して小さすぎないか判断する材料にする
        _, raw_counts = np.unique(masks, return_counts=True)
        raw_areas = raw_counts[1:]  # ラベル0(背景)を除く
        print(
            f"  Cellpose検出直後のROI面積(px): "
            f"最小{raw_areas.min()} / 中央値{int(np.median(raw_areas))} / "
            f"90パーセンタイル{int(np.percentile(raw_areas, 90))} / 最大{raw_areas.max()}"
            f"　→　この分布に対して換算後のMAX_AREA({max_area_px:.0f}px)が適切か確認してください"
        )

    masks = remove_large_masks(masks, max_area_px)
    n_after_area = int(masks.max())

    if APPLY_BRIGHTNESS_RATIO_FILTER:
        masks = filter_by_brightness_ratio(masks, img, MAX_BRIGHTNESS_RATIO)
    n_final = int(masks.max())

    if verbose:
        # 細胞数が想定より少ない場合、どの段階で減っているか確認するための内訳
        print(
            f"  内訳: Cellpose検出 {n_raw} 個 → 最大面積フィルタ後 {n_after_area} 個"
            + (f" → 輝度比フィルタ後 {n_final} 個" if APPLY_BRIGHTNESS_RATIO_FILTER else "")
        )

    return masks, flows


def masks_to_imagej_roi_zip(masks, save_path):
    """Cellposeのマスクを画像1枚分のImageJ RoiSet (.zip) として保存する
    (cellpose.io.save_rois と同じロジックだが、ファイル名の末尾に "_rois" を付けず、
    Fijiマクロが期待する "画像名.zip" そのままの名前で保存する)
    """
    outlines = utils.outlines_list(masks)
    rois = []
    for i, outline in enumerate(outlines):
        if len(outline) < 3:
            continue
        roi = roifile.ImagejRoi.frompoints(outline, name=f"{i + 1:04d}")
        rois.append(roi)

    save_path = Path(save_path)
    if save_path.exists():
        save_path.unlink()
    roifile.roiwrite(str(save_path), rois, mode="w")
    return len(rois)


## パラメータのプレビュー（1枚だけ試す）

一括処理の前に、まず1枚（1つのZスライス）でセグメンテーション結果を確認します。細胞が大きすぎる/小さすぎる、うまく分割できていない等があれば、上のパラメータ設定セルを調整してこのセルを再実行してください。


In [ ]:
preview_file = image_files[0]
preview_path = INPUT_DIR / preview_file
img = io.imread(str(preview_path))
print(f"対象ファイル: {preview_file}")
print(f"画像shape: {img.shape}, dtype: {img.dtype}")

masks, flows = segment_and_filter(img, preview_path, verbose=True)
print(f"フィルタ後: {masks.max()} 個の細胞を検出")

fig = plt.figure(figsize=(12, 5))
plot.show_segmentation(fig, img, masks, flows[0])
plt.tight_layout()
plt.show()


## 一括処理

`image_files` に含まれる全Zスライスに対して個別にセグメンテーション・フィルタ適用を行い、以下を保存します（Z投影はせず、1つのtifにつき1つのzipを作成します）。

- 元のtifと同じサブフォルダの `画像名.zip`: Fijiの `intensity_analysis.ijm` がそのまま読み込めるROI（細胞のみ、背景ROIは含みません）
- `QC_DIR` 以下（元と同じサブフォルダ構成）の `画像名_qc.png`: セグメンテーション結果を目視確認するための重ね合わせ画像

処理後、QC画像やFiji上でどのZが一番良いか確認し、使うZの `.zip` だけをFijiで開いて、おかしい細胞のROIを削除、背景ROIを最後に追加してから `.zip` を保存し直してください。


In [ ]:
results_summary = []

for rel_path in image_files:
    img_path = INPUT_DIR / rel_path
    img = io.imread(str(img_path))

    masks, flows = segment_and_filter(img, img_path, verbose=True)

    rel_dir = rel_path.parent
    basename = rel_path.stem

    # ROIは元のtifと同じサブフォルダに保存する（Fijiマクロの想定に合わせる）
    out_dir = OUTPUT_DIR / rel_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    roi_path = out_dir / f"{basename}.zip"
    n_rois = masks_to_imagej_roi_zip(masks, roi_path)

    # 目視確認用の重ね合わせ画像を保存（フォルダ構成をミラーする）
    qc_out_dir = QC_DIR / rel_dir
    qc_out_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img, cmap="gray")
    ax.imshow(np.ma.masked_where(masks == 0, masks), cmap="jet", alpha=0.5)
    ax.set_title(f"{rel_path} ({n_rois} cells)")
    ax.axis("off")
    fig.savefig(qc_out_dir / f"{basename}_qc.png", dpi=100, bbox_inches="tight")
    plt.close(fig)

    results_summary.append((str(rel_path), n_rois))
    print(f"{rel_path}: {n_rois} 個のROIを保存 -> {roi_path}")

print("\n=== 完了 ===")
for rel_path, n in results_summary:
    print(f"{rel_path}: {n} cells")


## 次のステップ (Fiji/ImageJ側の作業)

1. Google Drive for desktopなどでPCと同期するか、Driveから直接ダウンロードして、各撮影フォルダに `画像名.tif` と `画像名.zip` が同じフォルダにあることを確認する
2. `QC_DIR`（`qc_overlays`フォルダ、元と同じサブフォルダ構成）内の `_qc.png` を撮影ごとに見比べて、一番ピントが合っている・セグメンテーションが綺麗なZを選ぶ（崩れている場合はパラメータを調整して再実行）
3. Fijiで選んだZの画像を開き、ROI Managerで対応する `画像名.zip` を読み込む
4. 明らかにおかしい細胞のROIを選択して削除する（`APPLY_BRIGHTNESS_RATIO_FILTER` を無効にしている間は、この手動チェックが輝度比フィルタの代わりになります）
5. 背景となる領域のROIを新規作成し、**ROI Managerの一番最後に追加**する（`intensity_analysis.ijm` は最後のROIを背景として扱う仕様）
6. ROI Managerの内容を `画像名.zip` として上書き保存する
7. 従来通り `intensity_analysis.ijm` マクロを実行する（使わない他のZのtif/zipは無視してよい）
